# Data Preparation and Feature Engineering

In [1]:
import pandas as pd
import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.inspection import PartialDependenceDisplay




In [5]:
# Step 1: Paste the inlined CSV rows into data_str
data_str = """transaction_id,amount,account_age_months,credit_score,transaction_type,is_fraud
1001,4500.00,3,520,wire,1
1002,85.50,48,710,in-store,0
1003,12000.00,2,490,online,1
1004,230.00,60,780,in-store,0
1005,9800.00,1,505,wire,1
1006,310.00,55,740,online,0
1007,7600.00,4,530,wire,1
1008,150.00,72,800,in-store,0
1009,5400.00,3,515,online,1
1010,420.00,50,690,in-store,0
1011,3100.00,8,545,online,1
1012,95.00,44,760,in-store,0

"""
df = pd.read_csv(io.StringIO(data_str))
df.head()



,transaction_id,amount,account_age_months,credit_score,transaction_type,is_fraud
0,1001,4500.0,3,520,wire,1
1,1002,85.5,48,710,in-store,0
2,1003,12000.0,2,490,online,1
3,1004,230.0,60,780,in-store,0
4,1005,9800.0,1,505,wire,1


In [6]:
#Step 2: Prepare features
df = df.drop(columns=["transaction_id"])
df = pd.get_dummies(df, columns=["transaction_type"])
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [11]:
# Step 3: Unconstrained tree
dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train, y_train)
print("Unconstrained depth:", dt_unconstrained.get_depth())
print("Unconstrained leaves:", dt_unconstrained.get_n_leaves())


Unconstrained depth: 1
Unconstrained leaves: 2


In [12]:
# Step 4: Pruned tree
dt_pruned = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42)
dt_pruned.fit(X_train, y_train)
print("Pruned train accuracy:", accuracy_score(y_train, dt_pruned.predict(X_train)))
print("Pruned test accuracy:", accuracy_score(y_test, dt_pruned.predict(X_test)))


Pruned train accuracy: 0.5555555555555556
Pruned test accuracy: 0.3333333333333333


In [13]:

# Step 5: Feature importance
importances = pd.Series(dt_unconstrained.feature_importances_, index=X.columns)
print("Top 3 features:\n", importances.nlargest(3))

# Step 6: Partial dependence and interpretation
amount_idx = list(X.columns).index("amount")
PartialDependenceDisplay.from_estimator(dt_unconstrained, X_train, features=[amount_idx])
plt.savefig("pdp_amount.png"); plt.close()
print("Interpretation: As amount increases, predicted P(fraud) increases — high-value transactions carry higher fraud risk.")
print("Skeleton loaded — replace commented lines with your implementation.")

Top 3 features:
 amount                1.0
account_age_months    0.0
credit_score          0.0
dtype: float64
Interpretation: As amount increases, predicted P(fraud) increases — high-value transactions carry higher fraud risk.
Skeleton loaded — replace commented lines with your implementation.
